In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from tqdm import tqdm


In [2]:
IMG_SIZE = 70
BATCH_SIZE = 32
EPOCHS = 3
NUM_CLASSES = 3
KFOLD_SPLITS = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "../data/crops/c5_alto_grau"
OUTPUT_DIR = "../results/c5"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [3]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

In [4]:
dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
print("Classes:", dataset.classes)
print("Total de imagens:", len(dataset))

Classes: ['ASCH', 'HSIL', 'SCC']
Total de imagens: 3628


In [5]:
kf = KFold(n_splits=KFOLD_SPLITS, shuffle=True, random_state=42)
fold_metrics = []

for fold, (train_idx, val_idx) in enumerate(kf.split(dataset)):
    print(f"\n===== Fold {fold+1} =====")
    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)

    model = models.efficientnet_b0(weights="IMAGENET1K_V1")
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    model = model.to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0003)

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for inputs, labels in tqdm(train_loader, desc=f"Fold {fold+1} - Época {epoch+1}"):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1} - Loss médio: {running_loss/len(train_loader):.4f}")

    # Avaliação
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            y_pred.extend(preds.cpu().numpy())
            y_true.extend(labels.numpy())

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro")
    rec = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=dataset.classes, yticklabels=dataset.classes, cmap="Blues")
    plt.title(f"Matriz de Confusão - Fold {fold+1}")
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/confusao_fold{fold+1}.png")
    plt.close()

    fold_metrics.append({
        "fold": fold + 1,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1
    })

    print(f"Acurácia : {acc:.4f}")
    print(f"Precisão : {prec:.4f}")
    print(f"Revocação: {rec:.4f}")
    print(f"F1-Score : {f1:.4f}")



===== Fold 1 =====


Fold 1 - Época 1: 100%|██████████| 76/76 [00:16<00:00,  4.60it/s]


Epoch 1 - Loss médio: 0.7205


Fold 1 - Época 2: 100%|██████████| 76/76 [00:03<00:00, 21.20it/s]


Epoch 2 - Loss médio: 0.4534


...

Epoch 1 - Loss médio: 0.7299


Fold 3 - Época 2: 100%|██████████| 76/76 [00:03<00:00, 21.17it/s]


Epoch 2 - Loss médio: 0.4422


Fold 3 - Época 3: 100%|██████████| 76/76 [00:04<00:00, 18.26it/s]


Epoch 3 - Loss médio: 0.3406
Acurácia : 0.8602
Precisão : 0.8564
Revocação: 0.8617
F1-Score : 0.8588


In [6]:
df_metrics = pd.DataFrame(fold_metrics)
df_metrics.loc["média"] = df_metrics.mean(numeric_only=True)
df_metrics.to_csv(f"{OUTPUT_DIR}/metricas_classificador5.csv", index=False)
df_metrics


,fold,accuracy,precision,recall,f1_score
0,1.0,0.877686,0.877980,0.876729,0.877333
1,2.0,0.858561,0.858029,0.853390,0.855636
2,3.0,0.860215,0.856390,0.861736,0.858776
média,2.0,0.865487,0.864133,0.863952,0.863915
